<a href="https://colab.research.google.com/github/talhahahae/Deep-Learning/blob/HomeTaskWeek7/HomeWork_Week7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import lightning as pl
from torchvision.models import resnet18
import torch.nn as nn
import torch.optim as optim
import wandb
from lightning.pytorch.loggers import WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint

'''
Logs training details to Weights & Biases.
log_model="all": Logs all versions of the model.
project="ImageClassificationDemo": Names the project in WandB.
name='test2': Assigns a specific name to the run.
'''
wandb_logger = WandbLogger(log_model="all",project="ImageClassificationDemo",name='test2')

'''
transforms.Compose([...]): Chains multiple transformations together.
transforms.Resize((224, 224)): Resizes images to 224x224 pixels (needed for ResNet-18).
transforms.ToTensor(): Converts images to PyTorch tensors.
'''
# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

'''
torchvision.datasets.CIFAR10(...):
Loads the CIFAR-10 dataset.
root='./data': Specifies the directory to store the dataset.
train=True: Loads the training set.
train=False: Loads the test set.
download=True: Downloads the dataset if not already present.
transform=transform: Applies transformations (resize, tensor conversion).
'''
# Subset CIFAR-10 dataset
cifar10_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
cifar10_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

# Creates a subset of 1000 images from the training dataset.
# Total number of examples in the training set
total_samples = len(cifar10_train)
# Desired number of examples in the subset
subset_size = 1000
# Calculate the interval length
interval = total_samples // subset_size
print('interval', interval)
# Generate the subset indices
subset_indices = list(range(0, total_samples, interval))[:subset_size]
print('subsetindices', subset_indices)
# Create a subset of the CIFAR-10 training set
cifar10_train = torch.utils.data.Subset(cifar10_train, subset_indices)


# Total number of examples in the training set
total_samples = len(cifar10_test)
# Desired number of examples in the subset
subset_size = 500
# Calculate the interval length
interval = total_samples // subset_size
# Generate the subset indices
subset_indices = list(range(0, total_samples, interval))[:subset_size]
# Create a subset of the CIFAR-10 training set
cifar10_test = torch.utils.data.Subset(cifar10_test, subset_indices)


# Define data loaders
train_loader = DataLoader(cifar10_train, batch_size=2, shuffle=True)
test_loader = DataLoader(cifar10_test, batch_size=2, shuffle=False)

# Define the LightningModule
class ImageClassificationModel(pl.LightningModule):
    def __init__(self, num_classes, learning_rate=1e-3):
        super(ImageClassificationModel, self).__init__()
        self.learning_rate = learning_rate
        self.criterion = nn.CrossEntropyLoss()
        self.resnet = resnet18(pretrained=True)
        # Freeze all layers of ResNet except the custom head
        for param in self.resnet.parameters():
            param.requires_grad = False
        self.resnet.fc = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.resnet(x)
        return x

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        preds = torch.argmax(y_hat, dim=1)
        acc = (preds == y).float().mean()
        self.log('train/loss', loss,on_step=True,prog_bar=True,on_epoch=True)
        self.log('train/acc', acc,prog_bar=True,on_epoch=True,on_step=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        preds = torch.argmax(y_hat, dim=1)
        acc = (preds == y).float().mean()
        self.log('val/val_loss', loss,on_epoch=True,on_step=True,prog_bar=True)
        self.log('val/acc', acc, prog_bar=True,on_step=True,on_epoch=True)
        return loss

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=self.learning_rate)
        return optimizer

    def configure_callbacks(self):
        """Configures the ModelCheckpoint callback."""
        checkpoint_callback = ModelCheckpoint(
            monitor='val/acc',  # Monitor validation accuracy
            dirpath='./checkpoints',  # Directory to save checkpoints
            filename='best_model',  # Filename pattern
            save_top_k=1,  # Save only the best model
            mode='max',  # Save model with highest accuracy
            verbose=True
        )
        return [checkpoint_callback]  # Return a list of callbacks


# Train the model
model = ImageClassificationModel(num_classes=10)
callbacks = model.configure_callbacks()

trainer = pl.Trainer(logger=wandb_logger,max_epochs=5, devices=1, accelerator="auto",callbacks=callbacks)

trainer.fit(model, train_loader, test_loader)


100%|██████████| 170M/170M [00:02<00:00, 78.7MB/s]


interval 50
subsetindices [0, 50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600, 650, 700, 750, 800, 850, 900, 950, 1000, 1050, 1100, 1150, 1200, 1250, 1300, 1350, 1400, 1450, 1500, 1550, 1600, 1650, 1700, 1750, 1800, 1850, 1900, 1950, 2000, 2050, 2100, 2150, 2200, 2250, 2300, 2350, 2400, 2450, 2500, 2550, 2600, 2650, 2700, 2750, 2800, 2850, 2900, 2950, 3000, 3050, 3100, 3150, 3200, 3250, 3300, 3350, 3400, 3450, 3500, 3550, 3600, 3650, 3700, 3750, 3800, 3850, 3900, 3950, 4000, 4050, 4100, 4150, 4200, 4250, 4300, 4350, 4400, 4450, 4500, 4550, 4600, 4650, 4700, 4750, 4800, 4850, 4900, 4950, 5000, 5050, 5100, 5150, 5200, 5250, 5300, 5350, 5400, 5450, 5500, 5550, 5600, 5650, 5700, 5750, 5800, 5850, 5900, 5950, 6000, 6050, 6100, 6150, 6200, 6250, 6300, 6350, 6400, 6450, 6500, 6550, 6600, 6650, 6700, 6750, 6800, 6850, 6900, 6950, 7000, 7050, 7100, 7150, 7200, 7250, 7300, 7350, 7400, 7450, 7500, 7550, 7600, 7650, 7700, 7750, 7800, 7850, 7900, 7950, 8000, 8050, 8100, 8150, 8200, 8250, 

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 97.7MB/s]
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.util

<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

...

INFO: Epoch 47, global step 24000: 'val/acc' was not in top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 47, global step 24000: 'val/acc' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 48, global step 24500: 'val/acc' was not in top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 48, global step 24500: 'val/acc' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 49, global step 25000: 'val/acc' was not in top 1
INFO:lightning.pytorch.utilities.rank_zero:Epoch 49, global step 25000: 'val/acc' was not in top 1
INFO: `Trainer.fit` stopped: `max_epochs=50` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.


In [ ]:
!pip install lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.9/818.9 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.9/960.9 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.0/823.0 kB 44.2 MB/s eta 0:00:00


In [ ]:
!wandb login


wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: talhahahae (talhastinyasylum) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [ ]:
!pip install lightning


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.9/818.9 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.9/960.9 kB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import torchvision.models as models
from efficientnet_pytorch import EfficientNet

# Load ResNet50
resnet50 = models.resnet50(pretrained=True)
resnet50.eval()

# Create a random batch of input images (batch_size, channels, height, width)
# For ResNet, the input size is typically (3, 224, 224)
batch_size = 2  # Example batch size
input_tensor = torch.randn(batch_size, 3, 224, 224)

# Pass the random batch through ResNet
with torch.no_grad():  # Disable gradient calculation
    resnet_output = resnet50(input_tensor)

# Print output shape
print("ResNet50 Output Shape:", resnet_output.shape)


# Load MobileNetV2
mobilenet_v2 = models.mobilenet_v2(pretrained=True)
mobilenet_v2.eval()

# Create a random batch of input images
batch_size = 2  # Example batch size
input_tensor = torch.randn(batch_size, 3, 224, 224)

# Pass the random batch through MobileNet
with torch.no_grad():  # Disable gradient calculation
    mobilenet_output = mobilenet_v2(input_tensor)

# Print output shape
print("MobileNetV2 Output Shape:", mobilenet_output.shape)



# Load EfficientNetB0
efficientnet_b0 = EfficientNet.from_pretrained('efficientnet-b0')
efficientnet_b0.eval()

# Create a random batch of input images
batch_size = 2  # Example batch size
input_tensor = torch.randn(batch_size, 3, 224, 224)

# Pass the random batch through EfficientNet
with torch.no_grad():  # Disable gradient calculation
    efficientnet_output = efficientnet_b0(input_tensor)

# Print output shape
print("EfficientNetB0 Output Shape:", efficientnet_output.shape)

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 129MB/s]
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed i

ResNet50 Output Shape: torch.Size([2, 1000])


100%|██████████| 13.6M/13.6M [00:00<00:00, 109MB/s]
Downloading: "https://github.com/lukemelas/EfficientNet-PyTorch/releases/download/1.0/efficientnet-b0-355c32eb.pth" to /root/.cache/torch/hub/checkpoints/efficientnet-b0-355c32eb.pth


MobileNetV2 Output Shape: torch.Size([2, 1000])


100%|██████████| 20.4M/20.4M [00:00<00:00, 276MB/s]


Loaded pretrained weights for efficientnet-b0
EfficientNetB0 Output Shape: torch.Size([2, 1000])


In [ ]:
!pip install efficientnet_pytorch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 41.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 52.2 MB/s eta 0:00:00
  Created wheel for efficientnet_pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16424 sha256=988ff9304cb63e840b424dbc0fa82ddfd13b0d031ecda832c

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 17.0 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import pytorch_lightning as pl
import wandb
import optuna
from torch.utils.data import DataLoader, random_split
from pytorch_lightning.loggers import WandbLogger
from torchmetrics.classification import Accuracy

# Initialize Weights & Biases
wandb.init(project="transfer-learning-optuna")
wandb_logger = WandbLogger()

# Define Lightning Module
class TransferLearningModel(pl.LightningModule):
    def __init__(self, backbone: str = "resnet", lr: float = 1e-3):
        super().__init__()
        self.lr = lr
        self.backbone_name = backbone

        if backbone == "resnet":
            self.feature_extractor = models.resnet18(pretrained=True)
            self.feature_extractor.fc = nn.Linear(512, 10)
        elif backbone == "efficientnet":
            self.feature_extractor = models.efficientnet_b0(pretrained=True)
            self.feature_extractor.classifier[1] = nn.Linear(1280, 10)
        else:
            raise ValueError("Invalid backbone. Choose 'resnet' or 'efficientnet'")

        self.criterion = nn.CrossEntropyLoss()
        self.accuracy = Accuracy(task="multiclass", num_classes=10)  # Specify task type

    def forward(self, x):
        return self.feature_extractor(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = self.accuracy(y_hat, y)
        self.log("train_loss", loss)
        self.log("train_acc", acc)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        acc = self.accuracy(y_hat, y)
        self.log("val_loss", loss)
        self.log("val_acc", acc)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        acc = self.accuracy(y_hat, y)
        self.log("test_acc", acc)

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.lr)

# Load Dataset
def get_dataloaders(batch_size=32):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    dataset = datasets.CIFAR10(root="data", train=True, transform=transform, download=True)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
    test_dataset = datasets.CIFAR10(root="data", train=False, transform=transform, download=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)
    return train_loader, val_loader, test_loader

# Optuna Hyperparameter Optimization
def objective(trial):
    backbone = trial.suggest_categorical("backbone", ["resnet", "efficientnet"])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)

    model = TransferLearningModel(backbone=backbone, lr=learning_rate)
    train_loader, val_loader, _ = get_dataloaders()

    trainer = pl.Trainer(max_epochs=5, logger=wandb_logger, deterministic=True)
    trainer.fit(model, train_loader, val_loader)

    val_loss = trainer.callback_metrics["val_loss"].item()
    return val_loss

# Run Hyperparameter Search
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=10)

# Best Hyperparameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

# Train Final Model with Best Params
final_model = TransferLearningModel(**best_params)
train_loader, val_loader, test_loader = get_dataloaders()
trainer = pl.Trainer(max_epochs=10, logger=wandb_logger)
trainer.fit(final_model, train_loader, val_loader)
trainer.test(final_model, test_loader)

# Log Results
wandb.log({"best_params": best_params})
wandb.log({"final_model_test_acc": trainer.callback_metrics["test_acc"].item()})

# Finish Weights & Biases Run
wandb.finish()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: talhahahae (talhastinyasylum) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[I 2025-03-29 12:29:46,845] A new study created in memory with name: no-name-32de9d49-5941-4e98-8bee-6054ac02b108
<ipython-input-5-8edfc3a64233>:87: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get t

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

...

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.
[I 2025-03-29 16:23:38,440] Trial 3 finished with value: 0.5984595417976379 and parameters: {'backbone': 'resnet', 'learning_rate': 7.084319810966192e-05}. Best is trial 3 with value: 0.5984595417976379.
INFO:pytorch_lightning.utilities.rank_zero:You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.callbacks.model_summary:
  | Name              | Type               | Params | Mode 
-----------------------------------------------------------------
0 | feature_extractor | EfficientNet       | 4.0 M  | train
1 | criterion         | CrossEntropyLoss   | 0      | tr

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 13.4 MB/s eta 0:00:00
